# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JasperOwen/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding #2 The Content Performance Curve.** The paper claims that content peaks at 61-90 days, declines after 270 days and rebounds after a refresh. The label used in this finding is Health Score, a heuristic made up of impressions, search position, click-through rate and scroll depth. The finding groups the pages into age brackets and shows the average Health Score for each group

The data shows a comparison of pages of different ages instead of following the lifecycle of each individual page. The validation setup allows us to observe that there is a correlation between page age and lower Health Scores. Because of this, the methodology question I would ask is: Does the correlation between the age of a page and its performance indicate that age is the causation of performance?

Also, finding #2 does not state what subset of the data it uses. It could either be the full dataset or the active-content subset where impressions_90d and sessions_90d are both greater than 0. If finding 2 uses the active-content subset, it does not take into account pages that are older than 365 days and were refreshed, but did not recover from their decline. This could change the 365+ statistic in the table in find 2. Therefore I would ask which subset the table is drawn from.


**Finding #8: The Age-Freshness Matrix.** The paper claims that old content that gets refreshed performs nearly as well as new content. The labels used in this finding are age tiers and freshness tiers, which are created using timestamps from the data base and are used to group the pages. The other label used is Health Score, which is used to determine how well each page is performing.

While we can observe that refreshing old content gives it a similar Health Score to young and fresh content, the paper cannot prove that it was the refresh that was responsible for the improved Health Score. Therefore the methodology question I would ask is: Is a refresh directly responsible for an improvement in the performance of an older page?

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In the code cells below, I experimented with how I split my data: Having client data split across testing and training data (Random KFold) vs only having clients either in testing or training data (Client GroupK Fold).

With the Random KFold version of my model, the mean Precision@50 value was unrealistically high at 0.9. This is due to leakage coming from clients being spread across the training and testing data. It allows the model to learn client traits to identify them instead of observing if the clients' pages are potentially in decline.

With the Client GroupK Fold, my model's Precision@50 score drops to a more realistic value: 0.876. This happens because with clients only in either training or testing data, my model is forced to look at brand new clients that it has never seen before and determine if their pages are potentially in decline.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata
from huggingface_hub import login, hf_hub_download

# 1. Hugging Face Authentication & DuckDB Setup
hf_token = userdata.get("HF_TOKEN") if "google.colab" in sys.modules else os.environ.get("HF_TOKEN")
os.environ["HF_TOKEN"] = hf_token
login(token=hf_token)

con = duckdb.connect()
con.sql("SET enable_http_metadata_cache=true;")

file_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    filename="fact_content_daily_performance/month=2026-03/data_0.parquet",
    token=hf_token
)

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

In [5]:
page_performance_query = f"""
SELECT
    client_hash_id,
    content_hash_id,

    -- Week 1 (March 1-7) - Last Week
    SUM(CASE WHEN report_date <= '2026-03-07' THEN gsc_clicks ELSE 0 END) AS week1_clicks,
    SUM(CASE WHEN report_date <= '2026-03-07' THEN gsc_impressions ELSE 0 END) AS week1_impressions,
    AVG(CASE WHEN report_date <= '2026-03-07' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS week1_avg_position,

    -- Week 2 (March 8-15) - This Week
    SUM(CASE WHEN report_date > '2026-03-07' AND report_date <= '2026-03-15' THEN gsc_clicks ELSE 0 END) AS week2_clicks,
    SUM(CASE WHEN report_date > '2026-03-07' AND report_date <= '2026-03-15' THEN gsc_impressions ELSE 0 END) AS week2_impressions,
    AVG(CASE WHEN report_date > '2026-03-07' AND report_date <= '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS week2_avg_position,

    -- Late March (March 16-31) - The Future Target Outcome
    SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_clicks ELSE 0 END) AS late_clicks,
    SUM(CASE WHEN report_date > '2026-03-15' THEN gsc_impressions ELSE 0 END) AS late_impressions,
    AVG(CASE WHEN report_date > '2026-03-15' AND gsc_avg_position > 0 THEN gsc_avg_position ELSE NULL END) AS late_avg_position

FROM read_parquet('{file_path}')
WHERE gsc_data_available IS TRUE
  AND ga4_data_available IS TRUE
GROUP BY client_hash_id, content_hash_id
ORDER BY client_hash_id, content_hash_id
"""

df = con.sql(page_performance_query).df()
print(f"Loaded {len(df):,} rows from DuckDB.")

Loaded 63,856 rows from DuckDB.


In [6]:
def position_tier(pos):
    if pd.isna(pos) or pos == 0:
      return np.nan

    if pos <= 3:
        return 1
    elif pos <= 10:
        return 2
    elif pos <= 20:
        return 3
    elif pos <= 50:
        return 4
    else:
        return 5

df["w1_tier"] = df["week1_avg_position"].apply(position_tier)
df["w2_tier"] = df["week2_avg_position"].apply(position_tier)
df["late_tier"] = df["late_avg_position"].apply(position_tier)

# Historical tier change (Week 1 to Week 2, known by March 15th)
df["historical_tier_change"] = df["w2_tier"] - df["w1_tier"]

df.head()

,client_hash_id,content_hash_id,week1_clicks,week1_impressions,week1_avg_position,week2_clicks,week2_impressions,week2_avg_position,late_clicks,late_impressions,late_avg_position,w1_tier,w2_tier,late_tier,historical_tier_change
0,client_08d2847f24cf89c1,content_0071174ee1f89d8a,0.0,0.0,NaN,0.0,0.0,NaN,2.0,22.0,9.081197,NaN,NaN,2.0,NaN
1,client_08d2847f24cf89c1,content_07f1fdb8d2a0cf9d,3.0,143.0,9.724694,2.0,22.0,13.929825,2.0,21.0,17.807692,2.0,3.0,3.0,1.0
2,client_08d2847f24cf89c1,content_0f0c4d121b9d4aa5,0.0,0.0,NaN,0.0,1.0,4.000000,1.0,7.0,8.208333,NaN,2.0,2.0,NaN
3,client_08d2847f24cf89c1,content_12444e383a53d45f,0.0,0.0,NaN,0.0,0.0,NaN,0.0,3.0,19.000000,NaN,NaN,3.0,NaN
4,client_08d2847f24cf89c1,content_145bbdb1286836a0,0.0,0.0,NaN,0.0,0.0,NaN,0.0,2.0,10.000000,NaN,NaN,2.0,NaN


In [7]:
def visibility_bucket(impressions):
    if pd.isna(impressions):
        return np.nan
    elif impressions <= 8:
        return "low"
    elif impressions <= 459:
        return "medium"
    else:
        return "high"

df["visibility_bucket"] = df["week2_impressions"].apply(visibility_bucket)
df["visibility_score"] = df["visibility_bucket"].map({"low": 1, "medium": 2, "high": 4})

def calculate_baseline_score(row):
    if pd.notna(row["historical_tier_change"]) and row["historical_tier_change"] > 0:
        return row["historical_tier_change"] * row["visibility_score"]
    else:
        return 0.0

df["baseline_score"] = df.apply(calculate_baseline_score, axis=1)

model_df = df[df["w1_tier"].notna() & df["w2_tier"].notna() & df["late_tier"].notna()].copy().reset_index(drop=True)
model_df["future_decline"] = (model_df["late_tier"] > model_df["w1_tier"]).astype(int)

print(f"Dataset filtered: {len(model_df):,} rows with valid early positions.")

model_df.head()

Dataset filtered: 13,531 rows with valid early positions.


,client_hash_id,content_hash_id,week1_clicks,week1_impressions,week1_avg_position,week2_clicks,week2_impressions,week2_avg_position,late_clicks,late_impressions,late_avg_position,w1_tier,w2_tier,late_tier,historical_tier_change,visibility_bucket,visibility_score,baseline_score,future_decline
0,client_08d2847f24cf89c1,content_07f1fdb8d2a0cf9d,3.0,143.0,9.724694,2.0,22.0,13.929825,2.0,21.0,17.807692,2.0,3.0,3.0,1.0,medium,2,2.0,1
1,client_08d2847f24cf89c1,content_282a4afdf27795ee,0.0,2.0,19.500000,0.0,2.0,19.500000,0.0,5.0,31.083333,3.0,3.0,4.0,0.0,low,1,0.0,1
2,client_08d2847f24cf89c1,content_42cee104ce2fbeac,2.0,226.0,5.623577,6.0,475.0,6.634508,14.0,560.0,7.286641,2.0,2.0,2.0,0.0,high,4,0.0,0
3,client_08d2847f24cf89c1,content_7c7f399ab41e3f41,1.0,3.0,9.333333,1.0,5.0,10.800000,0.0,23.0,44.608696,2.0,3.0,4.0,1.0,low,1,1.0,1
4,client_08d2847f24cf89c1,content_907ea9ba9b787a4d,1.0,38.0,12.684211,0.0,16.0,10.875000,0.0,64.0,9.221218,3.0,3.0,2.0,0.0,medium,2,0.0,0


In [8]:
model_df["click_change_w1_w2"] = model_df["week2_clicks"] - model_df["week1_clicks"]
model_df["impression_change_w1_w2"] = model_df["week2_impressions"] - model_df["week1_impressions"]
model_df["log_w2_impressions"] = np.log1p(model_df["week2_impressions"])
model_df["log_w2_clicks"] = np.log1p(model_df["week2_clicks"])
model_df["w2_ctr"] = np.where(model_df["week2_impressions"] > 0, (model_df["week2_clicks"] / model_df["week2_impressions"]) * 100, 0.0)

feature_columns = [
    "week1_impressions", "week2_impressions", "week1_avg_position", "week2_avg_position",
    "w1_tier", "w2_tier", "historical_tier_change",
    "click_change_w1_w2", "impression_change_w1_w2",
    "log_w2_impressions", "log_w2_clicks", "w2_ctr"
]

model_df = model_df.reset_index(drop=True)

X = model_df[feature_columns].fillna(0)
y = model_df["future_decline"].values
clients = model_df["client_hash_id"].values
contents = model_df["content_hash_id"].values
baseline_scores = model_df["baseline_score"].values

In [9]:
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

def precision_at_k(y_true, scores, content_ids, k):
    eval_df = pd.DataFrame({
        'y': np.asarray(y_true, dtype=int),
        'score': np.asarray(scores, dtype=float),
        'content_hash_id': np.asarray(content_ids, dtype=str)
    })


    sorted_df = eval_df.sort_values(
        by=['score', 'content_hash_id'],
        ascending=[False, True]
    )

    top_k_labels = sorted_df.head(k)['y']
    return float(top_k_labels.mean())


In [10]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
gkf = GroupKFold(n_splits=5)

rf_model = RandomForestClassifier(n_estimators=100, max_depth=8, class_weight='balanced', random_state=42, n_jobs=1)

before_p50 = []
after_p50 = []


# Random K Fold
for train_idx, test_idx in kf.split(X, y):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    rf_model.fit(X_train, y_train)
    probs = rf_model.predict_proba(X_test)[:, 1]
    before_p50.append(precision_at_k(y_test, probs, contents[test_idx], k=50))



In [11]:
# Client GroupK Fold
for train_idx, test_idx in gkf.split(X, y, groups=clients):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    rf_model.fit(X_train, y_train)
    probs = rf_model.predict_proba(X_test)[:, 1]
    after_p50.append(precision_at_k(y_test, probs, contents[test_idx], k=50))

In [12]:
# Comparison table
split_comparison_df = pd.DataFrame({
    "Split Strategy": ["Random KFold (Before)", "Client GroupKFold (After)"],
    "Mean P@50": [np.mean(before_p50), np.mean(after_p50)],
    "Std P@50": [np.std(before_p50), np.std(after_p50)]
})

print("\nValidation Split Audit Results")
display(split_comparison_df.round(4))


Validation Split Audit Results


,Split Strategy,Mean P@50,Std P@50
0,Random KFold (Before),0.900,0.0580
1,Client GroupKFold (After),0.876,0.0843


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

#### Leakage check
1. **Time Boundaries:** All features only come from March 1-15 2026 in the dataset. All late March metrics are only used as evaluation labels.
2. **Correlation Audit:** The correlations between my features and my future_decline label are all between 0.41 and -0.41. As there are no unusually high correlations, this indicates that no future data leaked into my feature matrix.

#### Error examples
* **False Positives:** Pages like `content_e480902476969d77` went from high tiers to lowers teirs, causing the model to give them a high probability of them declining. However, these pages recovered in late March. This shows that my model can mistake short-term SERP turbulence for performance decay.
* **False Negatives** Pages like `content_2879ae1f678a6916` were stable between Week 1 and Week 2 with no tier change, causing them to be assignmed a low probability of their performance declining. however these pages did lose rankings in late March.
Because my model is leak-free, it cannot anticipate drops that show no early-window signal.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Checks each feature used by my model to make sure it is avaliable by my decision point: 15th March
# Records when the data came from, if it is safe or leaky and why.
leakage_audit = pd.DataFrame([
    ["week1_impressions", "March 1-7", "Safe", "Available before prediction"],
    ["week2_impressions", "March 8-15", "Safe", "Available before prediction"],
    ["week1_avg_position", "March 1-7", "Safe", "Available before prediction"],
    ["week2_avg_position", "March 8-15", "Safe", "Available before prediction"],
    ["w1_tier", "Derived from Week 1", "Safe", "Uses historical information only"],
    ["w2_tier", "Derived from Week 2", "Safe", "Uses information available by March 15"],
    ["historical_tier_change", "Week 1 → Week 2", "Safe", "Historical change only"],
    ["click_change_w1_w2", "Week 1 → Week 2", "Safe", "Historical change only"],
    ["impression_change_w1_w2", "Week 1 → Week 2", "Safe", "Historical change only"],
    ["log_w2_impressions", "Derived from Week 2", "Safe", "Transformation of historical feature"],
    ["log_w2_clicks", "Derived from Week 2", "Safe", "Transformation of historical feature"],
    ["w2_ctr", "Week 2", "Safe", "Calculated only from pre-prediction data"],
], columns=["Feature", "Information window", "Status", "Reason"])

display(leakage_audit)

,Feature,Information window,Status,Reason
0,week1_impressions,March 1-7,Safe,Available before prediction
1,week2_impressions,March 8-15,Safe,Available before prediction
2,week1_avg_position,March 1-7,Safe,Available before prediction
3,week2_avg_position,March 8-15,Safe,Available before prediction
4,w1_tier,Derived from Week 1,Safe,Uses historical information only
5,w2_tier,Derived from Week 2,Safe,Uses information available by March 15
6,historical_tier_change,Week 1 → Week 2,Safe,Historical change only
7,click_change_w1_w2,Week 1 → Week 2,Safe,Historical change only
8,impression_change_w1_w2,Week 1 → Week 2,Safe,Historical change only
9,log_w2_impressions,Derived from Week 2,Safe,Transformation of historical feature


In [14]:
# 2. Target correlation check
corr_with_target = model_df[feature_columns].apply(lambda col: col.corr(model_df["future_decline"]))
print("\nCorrelations of features with future_decline target:")
print(corr_with_target.round(4).to_string())

# Check no feature exceeds 0.9 or -0.9 correlation
assert not (corr_with_target.abs() > 0.9).any(), "High correlations, check for leakage"
print("\nLow correlations, leakage is unlikely")


rf_model.fit(X, y)
model_df["pred_prob"] = rf_model.predict_proba(X)[:, 1]
model_df["pred_label"] = (model_df["pred_prob"] >= 0.5).astype(int)



Correlations of features with future_decline target:
week1_impressions         -0.1004
week2_impressions         -0.1064
week1_avg_position        -0.2981
week2_avg_position        -0.1400
w1_tier                   -0.4008
w2_tier                   -0.1427
historical_tier_change     0.4044
click_change_w1_w2        -0.0218
impression_change_w1_w2   -0.0671
log_w2_impressions        -0.2047
log_w2_clicks             -0.0833
w2_ctr                     0.0669

Low correlations, leakage is unlikely


In [15]:

# False Positives: Model predicted decline but page did not decline
fp_examples = model_df[(model_df["future_decline"] == 0) & (model_df["pred_label"] == 1)].sort_values("pred_prob", ascending=False).head(2)

# False Negatives: Model predicted the page would not decline but it did
fn_examples = model_df[(model_df["future_decline"] == 1) & (model_df["pred_label"] == 0)].sort_values("pred_prob", ascending=True).head(2)

print("\nFalse Positive examples")
display(fp_examples[["client_hash_id", "content_hash_id", "historical_tier_change", "w1_tier", "w2_tier", "late_tier", "pred_prob", "future_decline"]])

print("\nFalse Negative examples")
display(fn_examples[["client_hash_id", "content_hash_id", "historical_tier_change", "w1_tier", "w2_tier", "late_tier", "pred_prob", "future_decline"]])


False Positive examples


,client_hash_id,content_hash_id,historical_tier_change,w1_tier,w2_tier,late_tier,pred_prob,future_decline
8944,client_3197e6291363b4db,content_e480902476969d77,1.0,1.0,2.0,1.0,0.960849,0
10956,client_e547b89c05043229,content_af64fcca906ce15f,1.0,1.0,2.0,1.0,0.960788,0



False Negative examples


,client_hash_id,content_hash_id,historical_tier_change,w1_tier,w2_tier,late_tier,pred_prob,future_decline
3110,client_23a62021009f63c4,content_2879ae1f678a6916,0.0,2.0,2.0,3.0,0.215271,1
488,client_20259bd6705d81d4,content_142e79e040389ec1,0.0,2.0,2.0,3.0,0.218752,1


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Before:** "Random Forest: By combining multiple feature signals (such as week1 rankings and ranking changes), my Random Forest consistently achieves the highest mean Precision@50 (0.876) out of all my models without data leakage."   
(Source: notebook w05_model, Section 4: Errors and interpretation)

**After:** Random Forest: Under a 5-fold Client GroupKFold validation split, my Random Forest model achieved a measured mean Precision@50 value of 0.876. This provides a ranked queue that can be used by editors to support their decisions when determining which pages to review for refresh first, rather than being used as definitive proof of decline in page performance

**Before:** Two of my Random Forest's top 3 features both measure the same information in different ways: w1_tier and week1_avg_position both measure a page's search ranking in week 1, with each feature making the other redundant. However the second most important feature: historical_tier_change measures the change in search tiers that a page goes through between weeks 1 and 2, which is ideal for my model. This means that my Random Forest model mainly decides based on each page's starting position and by how much it has already declined, which are good features for my model to work with.   
(Source: notebook w05_model, Section 4: Errors and interpretation)


**After:** In my Random Forest, I observed that the features with the largest weights were historical_tier_change (0.264717), week1_avg_position (0.252788) and w1_tier (0.184865). Because w1_tier and week1_avg_position capture different versions of the same information (a page's average search ranking in week 1), the model uses those features with a page's recent ranking changes as indicators of which direction they are moving in the search rankings. These measured signals are then used to prioritize pages for review for refresh. However, they only reflect the results of changes in the search algorithms or the page's performance rather than the search algorithm or page performance themselves.

In [16]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import json

audit_receipt = {
    "notebook": "w06_validation_audit.ipynb",
    "timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "random_split_p50": float(np.mean(before_p50)),
    "client_grouped_p50": float(np.mean(after_p50)),
    "generalization_gap": float(np.mean(before_p50) - np.mean(after_p50)),
    "leakage_audit_passed": True,
    "claim_rewrites_completed": True
}

os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/validation_audit_receipt.json", "w") as f:
    json.dump(audit_receipt, f, indent=2)

print("Validation audit receipt saved to work/outputs/validation_audit_receipt.json:")
print(json.dumps(audit_receipt, indent=2))

Validation audit receipt saved to work/outputs/validation_audit_receipt.json:
{
  "notebook": "w06_validation_audit.ipynb",
  "timestamp": "2026-08-17 09:54:50",
  "random_split_p50": 0.9,
  "client_grouped_p50": 0.876,
  "generalization_gap": 0.02400000000000002,
  "leakage_audit_passed": true,
  "claim_rewrites_completed": true
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.